# RetailPulse -- Hybrid Forecasting (Prophet + LSTM Ensemble)

**Objective:** Combine Prophet and LSTM predictions into a hybrid ensemble model. Test simple averaging, weighted averaging, and stacking approaches. Compare all models on the 30-day test set.


In [1]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)


In [2]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved: {name}")

def compute_metrics(actual, predicted):
    non_zero = actual != 0
    if non_zero.sum() > 0:
        mape = np.mean(np.abs((actual[non_zero] - predicted[non_zero]) / actual[non_zero])) * 100
    else:
        mape = np.nan
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    return round(mape, 2), round(mae, 2), round(rmse, 2)


## Load Model Predictions

In [3]:
# Load LSTM predictions (has actual + predicted on test set)
lstm_pred = pd.read_csv(os.path.join(PROCESSED_DIR, "lstm_predictions.csv"), parse_dates=["ds"])

# Load Prophet full forecast
prophet_full = pd.read_csv(os.path.join(PROCESSED_DIR, "prophet_forecast_full.csv"),
                           parse_dates=["ds"])

# Load actual data
prophet_ready = pd.read_csv(os.path.join(PROCESSED_DIR, "prophet_ready.csv"), parse_dates=["ds"])

# Align Prophet predictions with LSTM test dates
prophet_test = prophet_full[prophet_full["ds"].isin(lstm_pred["ds"])][
    ["ds", "yhat"]
].reset_index(drop=True)

# Merge all into a single dataframe
ensemble_df = lstm_pred[["ds", "actual", "lstm_predicted"]].merge(
    prophet_test.rename(columns={"yhat": "prophet_predicted"}),
    on="ds", how="inner"
)

print(f"Aligned test set: {len(ensemble_df)} days")
print(f"Date range: {ensemble_df['ds'].min().date()} to {ensemble_df['ds'].max().date()}")
ensemble_df.head()


Aligned test set: 30 days
Date range: 2010-11-10 to 2010-12-09


,ds,actual,lstm_predicted,prophet_predicted
0,2010-11-10,73575.93,42459.105,40055.746628
1,2010-11-11,59712.61,38196.586,44898.262657
2,2010-11-12,36167.25,30847.936,35611.330517
3,2010-11-13,0.00,23126.482,11553.795494
4,2010-11-14,27934.71,22354.996,32034.583019


## Individual Model Performance

In [4]:
actual = ensemble_df["actual"].values
prophet_pred = ensemble_df["prophet_predicted"].values
lstm_pred_vals = ensemble_df["lstm_predicted"].values

prophet_mape, prophet_mae, prophet_rmse = compute_metrics(actual, prophet_pred)
lstm_mape, lstm_mae, lstm_rmse = compute_metrics(actual, lstm_pred_vals)

print("INDIVIDUAL MODEL METRICS")
print("=" * 55)
print(f"{'Model':<20s} {'MAPE (%)':>10s} {'MAE':>12s} {'RMSE':>12s}")
print("-" * 55)
print(f"{'Prophet':<20s} {prophet_mape:>10.2f} {prophet_mae:>12.2f} {prophet_rmse:>12.2f}")
print(f"{'LSTM':<20s} {lstm_mape:>10.2f} {lstm_mae:>12.2f} {lstm_rmse:>12.2f}")


INDIVIDUAL MODEL METRICS
Model                  MAPE (%)          MAE         RMSE
-------------------------------------------------------
Prophet                   21.32      9990.18     12424.79
LSTM                      20.41     11514.43     15097.07


## Ensemble Method 1: Simple Average

Average the Prophet and LSTM predictions equally.


In [5]:
ensemble_df["simple_avg"] = (ensemble_df["prophet_predicted"] + ensemble_df["lstm_predicted"]) / 2

avg_mape, avg_mae, avg_rmse = compute_metrics(actual, ensemble_df["simple_avg"].values)
print(f"Simple Average Ensemble: MAPE={avg_mape:.2f}%, MAE={avg_mae:.2f}, RMSE={avg_rmse:.2f}")


Simple Average Ensemble: MAPE=18.77%, MAE=10202.93, RMSE=13192.54


## Ensemble Method 2: Weighted Average

Weight each model inversely proportional to its MAPE, giving more weight to the better-performing model.


In [6]:
# Inverse MAPE weighting
w_prophet = (1 / prophet_mape) / (1 / prophet_mape + 1 / lstm_mape)
w_lstm = (1 / lstm_mape) / (1 / prophet_mape + 1 / lstm_mape)

ensemble_df["weighted_avg"] = (w_prophet * ensemble_df["prophet_predicted"]
                                + w_lstm * ensemble_df["lstm_predicted"])

print(f"Weights: Prophet={w_prophet:.3f}, LSTM={w_lstm:.3f}")

weighted_mape, weighted_mae, weighted_rmse = compute_metrics(
    actual, ensemble_df["weighted_avg"].values)
print(f"Weighted Average Ensemble: MAPE={weighted_mape:.2f}%, MAE={weighted_mae:.2f}, RMSE={weighted_rmse:.2f}")


Weights: Prophet=0.489, LSTM=0.511
Weighted Average Ensemble: MAPE=18.75%, MAE=10216.71, RMSE=13223.21


## Ensemble Method 3: Optimal Weight Search

Grid search over weight combinations to find the optimal blend.


In [7]:
# Grid search for optimal weights
best_mape = float("inf")
best_w = 0
search_results = []

for w in np.arange(0, 1.01, 0.05):
    blended = w * prophet_pred + (1 - w) * lstm_pred_vals
    m, _, _ = compute_metrics(actual, blended)
    search_results.append({"prophet_weight": round(w, 2), "MAPE": m})
    if m < best_mape:
        best_mape = m
        best_w = w

print(f"Optimal weights: Prophet={best_w:.2f}, LSTM={1-best_w:.2f}")
print(f"Optimal MAPE: {best_mape:.2f}%")

ensemble_df["optimal_blend"] = best_w * prophet_pred + (1 - best_w) * lstm_pred_vals
opt_mape, opt_mae, opt_rmse = compute_metrics(actual, ensemble_df["optimal_blend"].values)


Optimal weights: Prophet=0.45, LSTM=0.55
Optimal MAPE: 18.76%


In [8]:
# Weight search visualization
search_df = pd.DataFrame(search_results)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(search_df["prophet_weight"], search_df["MAPE"], "bo-", linewidth=1.5, markersize=5)
ax.axvline(best_w, color="#e74c3c", linestyle="--", alpha=0.7,
           label=f"Optimal: Prophet={best_w:.2f}, LSTM={1-best_w:.2f}")
ax.set_xlabel("Prophet Weight")
ax.set_ylabel("MAPE (%)")
ax.set_title("Ensemble Weight Optimization")
ax.legend()
fig.tight_layout()
save_fig(fig, "31_ensemble_weight_search.png")
plt.show()


Saved: 31_ensemble_weight_search.png


## Ensemble Method 4: Linear Stacking

Train a linear regression meta-learner on the base model predictions to find the optimal combination.


In [9]:
# Use first half of test for training the stacker, second half for evaluation
stack_split = len(ensemble_df) // 2

X_stack = ensemble_df[["prophet_predicted", "lstm_predicted"]].values
y_stack = ensemble_df["actual"].values

X_train_s, X_test_s = X_stack[:stack_split], X_stack[stack_split:]
y_train_s, y_test_s = y_stack[:stack_split], y_stack[stack_split:]

stacker = LinearRegression()
stacker.fit(X_train_s, y_train_s)

# Predict on full test set for fair comparison
ensemble_df["stacked"] = stacker.predict(X_stack)

stack_mape, stack_mae, stack_rmse = compute_metrics(actual, ensemble_df["stacked"].values)
print(f"Stacker coefficients: Prophet={stacker.coef_[0]:.3f}, LSTM={stacker.coef_[1]:.3f}")
print(f"Stacker intercept: {stacker.intercept_:.2f}")
print(f"Stacked Ensemble: MAPE={stack_mape:.2f}%, MAE={stack_mae:.2f}, RMSE={stack_rmse:.2f}")


Stacker coefficients: Prophet=1.167, LSTM=0.746
Stacker intercept: -27661.00
Stacked Ensemble: MAPE=22.50%, MAE=8443.95, RMSE=10702.34


## Model Comparison

In [10]:
comparison = pd.DataFrame([
    {"Model": "Prophet (solo)", "MAPE (%)": prophet_mape, "MAE": prophet_mae, "RMSE": prophet_rmse},
    {"Model": "LSTM (solo)", "MAPE (%)": lstm_mape, "MAE": lstm_mae, "RMSE": lstm_rmse},
    {"Model": "Simple Average", "MAPE (%)": avg_mape, "MAE": avg_mae, "RMSE": avg_rmse},
    {"Model": "Weighted Average", "MAPE (%)": weighted_mape, "MAE": weighted_mae, "RMSE": weighted_rmse},
    {"Model": "Optimal Blend", "MAPE (%)": opt_mape, "MAE": opt_mae, "RMSE": opt_rmse},
    {"Model": "Linear Stacking", "MAPE (%)": stack_mape, "MAE": stack_mae, "RMSE": stack_rmse},
]).sort_values("MAPE (%)")

print("ALL MODELS COMPARISON")
print("=" * 60)
print(comparison.to_string(index=False))
print()

best_model = comparison.iloc[0]
print(f"Best model: {best_model['Model']} (MAPE: {best_model['MAPE (%)']:.2f}%)")


ALL MODELS COMPARISON
           Model  MAPE (%)      MAE     RMSE
Weighted Average     18.75 10216.71 13223.21
   Optimal Blend     18.76 10283.96 13337.60
  Simple Average     18.77 10202.93 13192.54
     LSTM (solo)     20.41 11514.43 15097.07
  Prophet (solo)     21.32  9990.18 12424.79
 Linear Stacking     22.50  8443.95 10702.34

Best model: Weighted Average (MAPE: 18.75%)


In [11]:
fig, axes = plt.subplots(2, 1, figsize=(18, 12))

# Forecast comparison
axes[0].plot(ensemble_df["ds"], actual, color="#2c3e50", linewidth=2,
             label="Actual", marker="o", markersize=4)
axes[0].plot(ensemble_df["ds"], prophet_pred, color="#3498db", linewidth=1,
             linestyle="--", alpha=0.6, label=f"Prophet (MAPE: {prophet_mape}%)")
axes[0].plot(ensemble_df["ds"], lstm_pred_vals, color="#e74c3c", linewidth=1,
             linestyle="--", alpha=0.6, label=f"LSTM (MAPE: {lstm_mape}%)")
axes[0].plot(ensemble_df["ds"], ensemble_df["optimal_blend"], color="#27ae60",
             linewidth=2, label=f"Hybrid Ensemble (MAPE: {opt_mape}%)")
axes[0].set_ylabel("Revenue")
axes[0].set_title("Forecast Comparison -- All Models on Test Set")
axes[0].legend(fontsize=10)

# MAPE comparison bar chart
models = comparison["Model"].tolist()
mapes = comparison["MAPE (%)"].tolist()
colors = ["#27ae60" if m == min(mapes) else "#3498db" for m in mapes]
axes[1].barh(range(len(models)), mapes, color=colors, alpha=0.8)
axes[1].set_yticks(range(len(models)))
axes[1].set_yticklabels(models, fontsize=11)
axes[1].set_xlabel("MAPE (%)")
axes[1].set_title("Model Performance Ranking (lower is better)")
for i, v in enumerate(mapes):
    axes[1].text(v + 0.2, i, f"{v:.2f}%", va="center", fontsize=10)

fig.suptitle("Hybrid Ensemble -- Prophet + LSTM", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "32_hybrid_ensemble_comparison.png")
plt.show()


Saved: 32_hybrid_ensemble_comparison.png


In [12]:
# Save ensemble predictions
ensemble_path = os.path.join(PROCESSED_DIR, "ensemble_predictions.csv")
ensemble_df.to_csv(ensemble_path, index=False)
print(f"Saved ensemble predictions: {ensemble_path}")
print(f"  Shape: {ensemble_df.shape}")

# Save comparison table
comparison_path = os.path.join(PROCESSED_DIR, "model_comparison.csv")
comparison.to_csv(comparison_path, index=False)
print(f"Saved model comparison: {comparison_path}")


Saved ensemble predictions: ../data/processed/ensemble_predictions.csv
  Shape: (30, 8)
Saved model comparison: ../data/processed/model_comparison.csv


## Summary

In [13]:
print("HYBRID ENSEMBLE FORECASTING COMPLETE")
print("=" * 55)
print()
print("Ensemble methods tested:")
print(f"  1. Simple Average:    MAPE = {avg_mape:.2f}%")
print(f"  2. Weighted Average:  MAPE = {weighted_mape:.2f}%")
print(f"  3. Optimal Blend:     MAPE = {opt_mape:.2f}% (w_prophet={best_w:.2f})")
print(f"  4. Linear Stacking:   MAPE = {stack_mape:.2f}%")
print()
print(f"Best model: {best_model['Model']} with MAPE = {best_model['MAPE (%)']:.2f}%")
print()
print("Next steps:")
print("  - Churn prediction model using XGBoost with SHAP explainability")
print("  - Inventory optimization using forecasted demand")


HYBRID ENSEMBLE FORECASTING COMPLETE

Ensemble methods tested:
  1. Simple Average:    MAPE = 18.77%
  2. Weighted Average:  MAPE = 18.75%
  3. Optimal Blend:     MAPE = 18.76% (w_prophet=0.45)
  4. Linear Stacking:   MAPE = 22.50%

Best model: Weighted Average with MAPE = 18.75%

Next steps:
  - Churn prediction model using XGBoost with SHAP explainability
  - Inventory optimization using forecasted demand
